Constructed examples demonstrate methods; fixed filing facts and figures are historical teaching inputs. New outputs are saved under a separate `runs/` directory. This does not reproduce a market strategy.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch01_lifecycle_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# Chapter 1: from a signal to net PnL

All values below are constructed teaching inputs. They are not market observations. Times use New York as a teaching convention: the signal at 16:05 follows the regular-session close, and the trade starts at the next 09:30 open. The return interval then spans the following 24 hours. This simplified schedule does not represent actual security trading. The calculation separates research output, executable holdings, trading cost, and realized net return.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quant_course import compute_returns

signal_time = "2024-01-02 16:05 America/New_York"
trade_time = "2024-01-03 09:30 America/New_York"
return_interval = "2024-01-03 09:30 to 2024-01-04 09:30 America/New_York"
holding_times = pd.date_range("2024-01-03 09:30", periods=2, freq="D", tz="America/New_York")
prices = pd.DataFrame({"asset_a": [100.0, 102.0], "asset_b": [100.0, 99.0]}, index=holding_times)
asset_returns = pd.Series({asset: compute_returns(prices[asset]).iloc[-1] for asset in prices})
print(f"signal_time={signal_time}; trade_time={trade_time}; return_interval={return_interval}")
asset_returns


## Gross and net budget

The risky holdings are 60% in asset A and 20% in asset B; the remaining 20% stays in cash with zero return. The trade increments total 20% gross traded notional (buys + sells, not halved). At 10 basis points per traded notional, transaction cost is 0.02% of NAV. The 100,000-unit starting capital is a teaching amount with no specified currency.

In [ ]:
weights = pd.Series({"asset_a": 0.6, "asset_b": 0.2})
cash_weight = 0.2
cash_return = 0.0
gross_return = float(weights.dot(asset_returns) + cash_weight * cash_return)
trade_increments = pd.Series({"asset_a": 0.1, "asset_b": -0.1})
total_traded_notional_ratio = float(trade_increments.abs().sum())
cost_rate = 10 / 10_000
trading_cost = total_traded_notional_ratio * cost_rate
net_return = gross_return - trading_cost
starting_nav = 100_000.0
ending_nav = starting_nav * (1 + net_return)
assert np.isclose(gross_return, 0.01)
assert np.isclose(trading_cost, 0.0002)
assert np.isclose(net_return, 0.0098)
assert np.isclose(ending_nav, 100_980.0)
pd.Series({"gross_return": gross_return, "total_traded_notional_ratio": total_traded_notional_ratio,
           "trading_cost": trading_cost, "net_return": net_return,
           "ending_NAV_capital_units": ending_nav})


In [ ]:
labels = ["Gross return", "Trading cost", "Net return"]
values = [gross_return * 100, -trading_cost * 100, net_return * 100]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, values, color=["#2878B5", "#D9534F", "#4C9F70"])
ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_ylabel("Return (%)")
ax.set_title("Simulated one-period gross and net return")
ax.bar_label(bars, fmt="%.2f%%")
fig.tight_layout()
output = RUN_DIR / "figures" / "a1" / "ch01_gross_net.png"
output.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output, dpi=160)
plt.close(fig)
output
